# CRÉATION D'UN RÉSEAU NEURONAL POUR LA PRÉDICTION DES PRIX DES VOITURES

Prétraitement des Données pour la Prédiction du Prix des Véhicules

In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# Load the dataset
file_path = '../../cleaned_data/cars_data_cleaned.csv'
data = pd.read_csv(file_path)

# Identify features and target
target = 'prix'  # Assuming 'prix' is the target variable
X = data.drop(columns=[target])
y = data[target]

# Identify numerical and categorical columns
numerical_columns = X.select_dtypes(include=['int64', 'float64']).columns
categorical_columns = X.select_dtypes(include=['object']).columns

# Preprocessing for numerical data: Imputation and Scaling
numerical_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Preprocessing for categorical data: Imputation and One-Hot Encoding
categorical_transformer = Pipeline(steps=[
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

# Combine preprocessors in a column transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_columns),
        ('cat', categorical_transformer, categorical_columns)
    ]
)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Apply transformations to training and testing data
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

# Output the shapes of the processed datasets
print("Shape of X_train:", X_train_transformed.shape)
print("Shape of X_test:", X_test_transformed.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)


Shape of X_train: (18361, 384)
Shape of X_test: (4591, 384)
Shape of y_train: (18361,)
Shape of y_test: (4591,)


Entraînement d'un Modèle de Régression pour la Prédiction du Prix des Véhicules avec TensorFlow

In [23]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Define the model
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train_transformed.shape[1],)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1)  # Single neuron for regression task
])

# Compile the model
model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Train the model
history = model.fit(
    X_train_transformed, y_train,
    validation_data=(X_test_transformed, y_test),
    epochs=50,
    batch_size=32,
    verbose=1
)

# Evaluate the model
loss, mae = model.evaluate(X_test_transformed, y_test, verbose=0)
print(f"Test Loss: {loss:.4f}, Test MAE: {mae:.4f}")


C:\Users\HP\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/50
574/574 ━━━━━━━━━━━━━━━━━━━━ 14s 12ms/step - loss: 12608179200.0000 - mae: 104859.8594 - val_loss: 6795490304.0000 - val_mae: 73608.9766
Epoch 2/50
574/574 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - loss: 4127109632.0000 - mae: 51675.1602 - val_loss: 990123648.0000 - val_mae: 23381.7344
Epoch 3/50
574/574 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - loss: 1043716800.0000 - mae: 24209.7852 - val_loss: 714353344.0000 - val_mae: 20041.9043
Epoch 4/50
574/574 ━━━━━━━━━━━━━━━━━━━━ 5s 8ms/step - loss: 789815168.0000 - mae: 20990.0801 - val_loss: 582906368.0000 - val_mae: 17742.1836
Epoch 5/50
574/574 ━━━━━━━━━━━━━━━━━━━━ 5s 9ms/step - loss: 684207680.0000 - mae: 19407.9961 - val_loss: 508806784.0000 - val_mae: 16355.1797
Epoch 6/50
574/574 ━━━━━━━━━━━━━━━━━━━━ 7s 12ms/step - loss: 639429312.0000 - mae: 18551.3906 - val_loss: 460898848.0000 - val_mae: 15513.1934
Epoch 7/50
574/574 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - loss: 576108928.0000 - mae: 17634.1758 - val_loss: 429018528.0000 - val_mae: 14948.

Évaluation des Performances du Modèle de Régression à l'aide des Métriques MSE, MAE et R²

In [25]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Faire des prédictions sur l'ensemble de test
y_test_pred = model.predict(X_test_transformed).flatten()

# Calculer les métriques
mse = mean_squared_error(y_test, y_test_pred)
mae = mean_absolute_error(y_test, y_test_pred)
r2 = r2_score(y_test, y_test_pred)

print(f"Mean Squared Error (MSE): {mse:.2f}")
print(f"Mean Absolute Error (MAE): {mae:.2f}")
print(f"Coefficient of Determination (R^2): {r2:.4f}")


144/144 ━━━━━━━━━━━━━━━━━━━━ 2s 14ms/step
Mean Squared Error (MSE): 205935885.94
Mean Absolute Error (MAE): 9983.13
Coefficient of Determination (R^2): 0.8717


Test du Modèle sur des Échantillons de Test et Comparaison des Prédictions avec les Valeurs Réelles

In [27]:
# Example: Test the model on a few samples from the test set
sample_indices = [0, 1, 2]  # Replace with desired indices or use random sampling
X_sample = X_test.iloc[sample_indices]
y_sample_actual = y_test.iloc[sample_indices]

# Preprocess the sample data
X_sample_transformed = preprocessor.transform(X_sample)

# Make predictions
y_sample_pred = model.predict(X_sample_transformed)

# Compare predictions with actual values
for i, (pred, actual) in enumerate(zip(y_sample_pred.flatten(), y_sample_actual)):
    print(f"Sample {i+1}: Predicted Price = {pred:.2f}, Actual Price = {actual:.2f}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 373ms/step
Sample 1: Predicted Price = 146960.70, Actual Price = 155000.00
Sample 2: Predicted Price = 77059.34, Actual Price = 83000.00
Sample 3: Predicted Price = 66932.43, Actual Price = 70000.00


Prédiction du Prix pour de Nouvelles Données à l'Aide du Modèle Entraîné

In [29]:
# Example new data
new_data = pd.DataFrame({
    'marque': ['opel'],
    'modele': ['corsa'],
    'annee-modele': [2013],
    'kilometrage': [157500],
    'type_de_carburant': ['essence'],
    'puissance_fiscale': [9],
    'boite_de_vitesses': ['manuelle'],
    'nombre_de_portes': [6],
    'origine': ['ww au maroc'],
    'premiere_main': [0],
    'etat': ['tres bon'],
    'airbags': [2],
    'climatisation': [1],
    'abs': [1],
    'esp': [1],
    'cd/mp3/bluetooth': [1]
})

# Preprocess new data
new_data_transformed = preprocessor.transform(new_data)

# Predict
new_predictions = model.predict(new_data_transformed)
print(f"Predicted Price for New Data: {new_predictions[0][0]:.2f}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 382ms/step
Predicted Price for New Data: 69408.14
